In [1]:
import os
from dotenv import load_dotenv

print("Thư mục hiện tại:", os.getcwd())

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    print("Đã chuyển thư mục ra root:", os.getcwd())

load_dotenv(override=True)
print("Đã load file .env thành công!")

Thư mục hiện tại: d:\code\VietCulture_Rag\notebooks
Đã chuyển thư mục ra root: d:\code\VietCulture_Rag
Đã load file .env thành công!


In [2]:
import json
import pandas as pd
from tqdm import tqdm
import time
from src.agent.graph import create_agent_bundle, invoke_agent 
from src.retrieval import qa_retriever
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
# import json
# import random
# import time
# import pandas as pd
# from tqdm import tqdm
# from src.agent.graph import create_agent_bundle, invoke_agent

# bundle = create_agent_bundle()

# with open("notebooks/output_benchmark.json") as f:
#     data = json.load(f)

# results = []
# random.seed(42)  

# for sample in tqdm(data[:1], desc="Đang đánh giá"):
#     # Lấy ngẫu nhiên 1 trong 3 queries
#     user_queries = sample.get("user_query", [])
#     if not user_queries:
#         continue
    
#     q = random.choice(user_queries)   # ← bốc ngẫu nhiên 1
#     user_query = q["text"]
#     query_id   = q["query_id"]
#     conv_id    = f"eval_{sample.get('benchmark_id', '0')}_{query_id}"

#     # Retrieve
#     retrieved_docs  = bundle.retriever.retrieve(query=user_query)
#     retrieved_texts = [chunk.document.page_content 
#                        for chunk in retrieved_docs]
#     retrieved_ids   = [chunk.document.metadata.get("doc_id", "") 
#                        for chunk in retrieved_docs]

#     print(f"\n[{sample['benchmark_id']} - {query_id}] {user_query}")

#     # Generate answer
#     try:
#         graph_response = invoke_agent(
#             bundle=bundle,
#             user_id="evaluator_user",
#             conversation_id=conv_id,
#             message=user_query
#         )
#         llm_answer = graph_response.get("answer", "")
#     except Exception as e:
#         print(f"Lỗi ở {sample['benchmark_id']} - {query_id}: {e}")
#         llm_answer = ""

#     results.append({
#         "benchmark_id"      : sample.get("benchmark_id", ""),
#         "query_id"          : query_id,
#         "category"          : sample.get("category", ""),
#         "difficulty"        : sample.get("difficulty", "unknown"),
#         "question"          : user_query,
#         "answer"            : llm_answer,        # ← thống nhất tên
#         "dataset_answer"    : sample.get("answer", ""),
#         "contexts"          : retrieved_texts,
#         "ground_truth"      : sample.get("detailed_explanation", ""),
#         "retrieved_ids"     : retrieved_ids,
#         "ground_truth_chunk": sample.get("ground_truth_chunk_id", "")
#     })

#     time.sleep(4)

# # Lưu kết quả
# pd.DataFrame(results).to_json(
#     "notebooks/eval_results_test1.json",   # ← đổi tên tránh ghi đè
#     orient="records",
#     force_ascii=False
# )
# print(f"Done! {len(results)} samples")

In [4]:
# df = pd.read_json("notebooks/eval_results_test3.json")

# def is_hit(gt_id, retrieved_ids, k):
#     gt_parts = gt_id.split(":")
#     if len(gt_parts) < 5: return False
    
#     gt_cat = gt_parts[0]
#     gt_kw = gt_parts[1]
#     gt_question = ":".join(gt_parts[4:]) # Lấy phần câu hỏi ở cuối
    
#     for ret_id in retrieved_ids[:k]:
#         ret_parts = ret_id.split(":")
#         if len(ret_parts) < 5: continue
        
#         ret_cat = ret_parts[0]
#         ret_kw = ret_parts[1]
#         ret_question = ":".join(ret_parts[4:])
        
#         # Nếu trùng chủ đề + từ khóa + câu hỏi là được vì nhiều ảnh có thể cùng 1 câu hỏi vì thế sẽ cho hit rate = 0
#         if gt_cat == ret_cat and gt_kw == ret_kw and gt_question == ret_question:
#             return True
            
#     return False

# def hit_rate(df, k):
#     hits = 0
#     for _, row in df.iterrows():
#         gt_id = row["ground_truth_chunk"]
#         retrieved_ids = row["retrieved_ids"]
        
#         if is_hit(gt_id, retrieved_ids, k):
#             hits += 1
            
#     return hits / len(df) if len(df) > 0 else 0

# print("--- Retrieval Metrics ---")
# print(f"Hit Rate @1 : {hit_rate(df, 1):.3f}")
# print(f"Hit Rate @3 : {hit_rate(df, 3):.3f}")
# print(f"Hit Rate @5 : {hit_rate(df, 5):.3f}")

# print("\n--- Breakdown theo Category ---")
# for cat, group in df.groupby("category"):
#     print(f"{cat:25s} : {hit_rate(group, 5):.3f}")

# print("\n--- Breakdown theo Difficulty ---")
# for diff, group in df.groupby("difficulty"):
#     print(f"{diff:10s} : {hit_rate(group, 5):.3f}")

Vì sử dụng dataset là 240 sample được viết lại câu hỏi sao cho không đổi ý nghĩa nên nếu tính hit rate theo doc_id sẽ cho ra kết quả rất tệ, vì các câu hỏi này đã thay đổi các từ ngữ rồi. Nên nhóm tính hit rate dựa trên khớp category và keyword để xem chatbot có truy xuất đúng chủ đề mà người dùng yêu cầu hay không

In [5]:
# for _, row in df.iterrows():
#     gt_id = row["ground_truth_chunk"]
#     retrieved_ids = row["retrieved_ids"]
    
#     if not is_hit(gt_id, retrieved_ids, 1):
#         print(f"Câu hỏi: {row['question']}")
#         print(f"Đáp án chuẩn : {gt_id}")
        
#         if retrieved_ids:
#             print(f"Bot lấy về 1 : {retrieved_ids[0]}")
#             if len(retrieved_ids) > 1:
#                 print(f"Bot lấy về 2 : {retrieved_ids[1]}")
#         else:
#             print("Bot lấy về  : Không tìm thấy gì")
#         print("-" * 50)

In [6]:
# import pandas as pd

# df = pd.read_json("notebooks/eval_results_test3.json") 

# def hit_rate(df, k):
#     hits = 0
#     for _, row in df.iterrows():
#         gt_id = str(row.get("ground_truth_chunk", ""))
#         retrieved_ids = row.get("retrieved_ids", [])
        
#         gt_parts = gt_id.split(":")
#         if len(gt_parts) < 2: 
#             continue
#         gt_prefix = f"{gt_parts[0]}:{gt_parts[1]}"
        
#         match = False
#         for ret_id in retrieved_ids[:k]:
#             ret_parts = str(ret_id).split(":")
#             if len(ret_parts) >= 2:
#                 ret_prefix = f"{ret_parts[0]}:{ret_parts[1]}"
                
#                 if gt_prefix == ret_prefix:
#                     match = True
#                     break
        
#         if match:
#             hits += 1
            
#     return hits / len(df) if len(df) > 0 else 0

# print("--- Retrieval Metrics ---")
# print(f"Hit Rate @1 : {hit_rate(df, 1):.3f}")
# print(f"Hit Rate @3 : {hit_rate(df, 3):.3f}")
# print(f"Hit Rate @5 : {hit_rate(df, 5):.3f}")

# print("\n--- Breakdown theo Category ---")
# for cat, group in df.groupby("category"):
#     print(f"{cat:25s} : {hit_rate(group, 5):.3f} ({len(group)} samples)")

# print("\n--- Breakdown theo Difficulty ---")
# for diff, group in df.groupby("difficulty"):
#     print(f"{diff:10s} : {hit_rate(group, 5):.3f} ({len(group)} samples)")

In [ ]:
import json
import random
import time
import pandas as pd
from tqdm import tqdm
from src.agent.graph import create_agent_bundle, invoke_agent

bundle = create_agent_bundle()

with open("data/output_benchmark.json") as f:
    data = json.load(f)

results = []
random.seed(42)  

for sample in tqdm(data, desc="Đang đánh giá"):
    # Lấy ngẫu nhiên 1 trong 3 queries
    user_queries = sample.get("user_query", [])
    if not user_queries:
        continue
    
    q = random.choice(user_queries)   # bốc ngẫu nhiên 1
    user_query = q["text"]
    query_id   = q["query_id"]
    conv_id    = f"eval_{sample.get('benchmark_id', '0')}_{query_id}"

    # Retrieve
    retrieved_docs  = bundle.retriever.retrieve(query=user_query)
    retrieved_texts = [chunk.document.page_content 
                       for chunk in retrieved_docs]
    retrieved_ids   = [chunk.document.metadata.get("doc_id", "") 
                       for chunk in retrieved_docs]

    print(f"\n[{sample['benchmark_id']} - {query_id}] {user_query}")

    # Generate answer
    try:
        graph_response = invoke_agent(
            bundle=bundle,
            user_id="evaluator_user",
            conversation_id=conv_id,
            message=user_query
        )
        llm_answer = graph_response.get("answer", "")
    except Exception as e:
        print(f"Lỗi ở {sample['benchmark_id']} - {query_id}: {e}")
        llm_answer = ""

    results.append({
        "benchmark_id"      : sample.get("benchmark_id", ""),
        "query_id"          : query_id,
        "category"          : sample.get("category", ""),
        "difficulty"        : sample.get("difficulty", "unknown"),
        "question"          : user_query,
        "answer"            : llm_answer,        # ← thống nhất tên
        "dataset_answer"    : sample.get("answer", ""),
        "contexts"          : retrieved_texts,
        "ground_truth"      : sample.get("detailed_explanation", ""),
        "retrieved_ids"     : retrieved_ids,
        "ground_truth_chunk": sample.get("ground_truth_chunk_id", "")
    })

    time.sleep(10)

# Lưu kết quả
pd.DataFrame(results).to_json(
    "notebooks/eval_results_240.json",  
    orient="records",
    force_ascii=False
)
print(f"Done! {len(results)} samples")

E0000 00:00:1782539140.976611   87230 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1782539140.976630   87230 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1782539140.976631   87230 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1782539140.976632   87230 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1782539140.976633   87230 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Đang đánh giá:   0%|          | 0/240 [00:00<?, ?it/s]


[bench_0001 - q3] Ý nghĩa văn hóa của bánh bao trong nền ẩm thực truyền thống Việt Nam là gì?


Đang đánh giá:   0%|          | 1/240 [00:25<1:39:53, 25.08s/it]

...


[bench_0239 - q3] Theo bạn, tại sao Ca trù được coi là quan trọng trong văn hóa Việt Nam?


Đang đánh giá: 100%|█████████▉| 239/240 [1:45:42<00:26, 26.13s/it]


[bench_0240 - q3] Tầm ảnh hưởng của nghệ thuật cải lương đến văn hóa xã hội ở Việt Nam là gì?


Đang đánh giá: 100%|██████████| 240/240 [1:46:06<00:00, 26.53s/it]

Done! 240 samples


In [2]:
import pandas as pd

df = pd.read_json("notebooks/eval_results_240.json") 

def hit_rate(df, k):
    hits = 0
    for _, row in df.iterrows():
        gt_id = str(row.get("ground_truth_chunk", ""))
        retrieved_ids = row.get("retrieved_ids", [])
        
        gt_parts = gt_id.split(":")
        if len(gt_parts) < 2: 
            continue
        gt_prefix = f"{gt_parts[0]}:{gt_parts[1]}"
        
        match = False
        for ret_id in retrieved_ids[:k]:
            ret_parts = str(ret_id).split(":")
            if len(ret_parts) >= 2:
                ret_prefix = f"{ret_parts[0]}:{ret_parts[1]}"
                
                if gt_prefix == ret_prefix:
                    match = True
                    break
        
        if match:
            hits += 1
            
    return hits / len(df) if len(df) > 0 else 0

print("--- Retrieval Metrics ---")
print(f"Hit Rate @1 : {hit_rate(df, 1):.3f}")
print(f"Hit Rate @3 : {hit_rate(df, 3):.3f}")
print(f"Hit Rate @5 : {hit_rate(df, 5):.3f}")

print("\n--- Breakdown theo Category ---")
for cat, group in df.groupby("category"):
    print(f"{cat:25s} : {hit_rate(group, 5):.3f} ({len(group)} samples)")

print("\n--- Breakdown theo Difficulty ---")
for diff, group in df.groupby("difficulty"):
    print(f"{diff:10s} : {hit_rate(group, 5):.3f} ({len(group)} samples)")

--- Retrieval Metrics ---
Hit Rate @1 : 0.487
Hit Rate @3 : 0.600
Hit Rate @5 : 0.646

--- Breakdown theo Category ---
am_thuc                   : 0.850 (20 samples)
doi_song_hang_ngay        : 0.600 (20 samples)
giao_thong                : 0.650 (20 samples)
kien_truc                 : 0.600 (20 samples)
le_hoi                    : 0.550 (20 samples)
nhac_cu                   : 0.500 (20 samples)
phong_canh                : 0.850 (20 samples)
the_thao_truyen_thong     : 0.800 (20 samples)
thu_cong_my_nghe          : 0.650 (20 samples)
trang_phuc                : 0.500 (20 samples)
tro_choi_dan_gian         : 0.550 (20 samples)
van_hoa_dan_gian          : 0.650 (20 samples)

--- Breakdown theo Difficulty ---
hard       : 0.757 (107 samples)
medium     : 0.556 (133 samples)


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

eval_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_retries=3,
    timeout=60,
)

response = eval_llm.invoke("Xin chào, trả lời ngắn thôi")
print(response.content)
print("✅ API còn quota")

Chào.
✅ API còn quota


In [10]:
if os.path.exists("notebooks/ragas_checkpoint.csv"):
    os.remove("notebooks/ragas_checkpoint.csv")

In [11]:
import os
import pandas as pd
import time
from dotenv import load_dotenv
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings



eval_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_retries=3,
    timeout=60,
)

eval_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base"
)

df_results = pd.read_json("notebooks/eval_results_240.json") 
print(f"Tổng samples: {len(df_results)}")  

CHECKPOINT = "notebooks/ragas_checkpoint.csv"
BATCH_SIZE = 10
SLEEP_SEC  = 30

if os.path.exists(CHECKPOINT):
    df_done    = pd.read_csv(CHECKPOINT)
    done_count = len(df_done)
    all_scores = [df_done]
    print(f"Resume từ sample {done_count}")
else:
    all_scores = []
    done_count = 0
    print("Bắt đầu từ đầu")

df_remaining = df_results.iloc[done_count:].reset_index(drop=True)
total        = len(df_remaining)
print(f"Còn lại: {total} samples\n")

for i in range(0, total, BATCH_SIZE):
    batch     = df_remaining.iloc[i:i+BATCH_SIZE].copy()
    batch_num = (done_count + i) // BATCH_SIZE + 1
    n_end     = done_count + i + len(batch)

    print(f"{'─'*50}")
    print(f"Batch {batch_num}: samples {done_count+i+1} → {n_end}")

    try:
        batch["ground_truth"] = (
            batch["ground_truth"] + " " + batch["dataset_answer"]
        ).str.strip()

        dataset = Dataset.from_pandas(
            batch[["question", "answer", 
                   "contexts", "ground_truth"]]
        )
        score = evaluate(
            dataset=dataset,
            metrics=[faithfulness, answer_relevancy],
            llm=eval_llm,
            embeddings=eval_embeddings,
            raise_exceptions=False,
        )
        df_batch = score.to_pandas()
        all_scores.append(df_batch)
        pd.concat(all_scores, ignore_index=True).to_csv(
            CHECKPOINT, index=False
        )
        faith = df_batch["faithfulness"].mean()
        rel   = df_batch["answer_relevancy"].mean()
        print(f"faith={faith:.3f} | relevancy={rel:.3f}")

    except Exception as e:
        print(f"Lỗi: {e}")
        print(f"Nghỉ 60s rồi thử lại")
        time.sleep(60)
        try:
            score = evaluate(
                dataset=dataset,
                metrics=[faithfulness, answer_relevancy],
                llm=eval_llm,
                embeddings=eval_embeddings,
                raise_exceptions=False,
            )
            df_batch = score.to_pandas()
            all_scores.append(df_batch)
            pd.concat(all_scores, ignore_index=True).to_csv(
                CHECKPOINT, index=False
            )
            print(f"Retry OK")
        except Exception as e2:
            print(f"Bỏ qua batch này: {e2}")

    if i + BATCH_SIZE < total:
        print(f"💤 Nghỉ {SLEEP_SEC}s...")
        time.sleep(SLEEP_SEC)

if all_scores:
    df_final = pd.concat(all_scores, ignore_index=True)
    df_final.to_csv("notebooks/ragas_240_score.csv", index=False) 

    print(f"\n{'═'*50}")
    print(f"TỔNG KẾT ({len(df_final)}/240 samples)")  
    print(f"{'═'*50}")
    print(f"Faithfulness    : {df_final['faithfulness'].mean():.3f}")
    print(f"Answer Relevancy: {df_final['answer_relevancy'].mean():.3f}")

    # Breakdown theo category
    df_merged = df_results.iloc[:len(df_final)].copy()
    df_merged["faithfulness"]     = df_final["faithfulness"].values
    df_merged["answer_relevancy"] = df_final["answer_relevancy"].values

    print(f"\n── Theo Category ──")
    for cat, g in df_merged.groupby("category"):
        print(f"  {cat:25s}: "
              f"faith={g['faithfulness'].mean():.3f} | "
              f"rel={g['answer_relevancy'].mean():.3f}")

    print(f"\n── Theo Difficulty ──")
    for diff, g in df_merged.groupby("difficulty"):
        print(f"  {diff:10s}: "
              f"faith={g['faithfulness'].mean():.3f} | "
              f"rel={g['answer_relevancy'].mean():.3f}")

/tmp/ipykernel_87230/2472571070.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_87230/2472571070.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Tổng samples: 240
Bắt đầu từ đầu
Còn lại: 240 samples

──────────────────────────────────────────────────
Batch 1: samples 1 → 10


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


faith=0.726 | relevancy=0.806
💤 Nghỉ 30s...
──────────────────────────────────────────────────
Batch 24: samples 231 → 240


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


faith=0.921 | relevancy=0.835

══════════════════════════════════════════════════
TỔNG KẾT (240/240 samples)
══════════════════════════════════════════════════
Faithfulness    : 0.741
Answer Relevancy: 0.787

── Theo Category ──
  am_thuc                  : faith=0.706 | rel=0.722
  doi_song_hang_ngay       : faith=0.789 | rel=0.731
  giao_thong               : faith=0.707 | rel=0.844
  kien_truc                : faith=0.687 | rel=0.747
  le_hoi                   : faith=0.822 | rel=0.791
  nhac_cu                  : faith=0.766 | rel=0.898
  phong_canh               : faith=0.660 | rel=0.747
  the_thao_truyen_thong    : faith=0.611 | rel=0.697
  thu_cong_my_nghe         : faith=0.858 | rel=0.888
  trang_phuc               : faith=0.856 | rel=0.839
  tro_choi_dan_gian        : faith=0.606 | rel=0.727
  van_hoa_dan_gian         : faith=0.824 | rel=0.820

── Theo Difficulty ──
  hard      : faith=0.767 | rel=0.825
  medium    : faith=0.719 | rel=0.757
